# 04 — Header Inclusion Analysis
**Purpose:** Identify high-impact headers for refactoring to reduce build times.
**Inputs:** `header_edges.parquet`, `header_metrics.parquet`, `file_metrics.parquet`,
            `git_commit_log.parquet` (optional)
**Outputs:** `data/intermediate/header_impact.parquet`,
             `data/intermediate/gephi/include_graph.gexf`

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from buildanalysis.graphs import build_include_graph
from buildanalysis.headers import (
    compute_header_impact_score,
    compute_header_pagerank,
    compute_include_amplification,
    compute_include_fan_metrics,
)
from buildanalysis.export import export_include_graph
from buildanalysis.git import compute_file_churn
from buildanalysis.loading import BuildDataset

warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 100})

DATA_DIR = Path("../../data/processed")
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)

# Check for optional git data
HAS_GIT = ds.has_file("git_commit_log")
if HAS_GIT:
    print("Git commit log found — churn data will be included in impact scoring.")
else:
    print("Git commit log not found — proceeding without churn data.")

print(f"header_edges:   {ds.header_edges.shape[0]:,} rows")
print(f"header_metrics: {ds.header_metrics.shape[0]:,} rows")
print(f"file_metrics:   {ds.file_metrics.shape[0]:,} rows")

## 1. Build the Include Graph

In [ ]:
import networkx as nx

include_graph = build_include_graph(ds.header_edges)

n_nodes = include_graph.number_of_nodes()
n_edges = include_graph.number_of_edges()

# Remove system headers for component analysis
non_system = include_graph.copy()
system_nodes = [n for n, d in non_system.nodes(data=True) if d.get("is_system", False)]
non_system.remove_nodes_from(system_nodes)

n_components = nx.number_weakly_connected_components(non_system)

print(f"Include graph: {n_nodes:,} nodes, {n_edges:,} edges")
print(f"  System headers:        {len(system_nodes):,}")
print(f"  Project files:         {n_nodes - len(system_nodes):,}")
print(f"  Connected components:  {n_components}")

## 2. Fan-In / Fan-Out Analysis

In [ ]:
fan_metrics = compute_include_fan_metrics(include_graph)

headers = fan_metrics[fan_metrics["is_header"]].copy()
sources = fan_metrics[~fan_metrics["is_header"]].copy()

print(f"Headers: {len(headers):,}")
print(f"Sources: {len(sources):,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fan-in distribution for headers
ax = axes[0]
ax.hist(headers["direct_fan_in"], bins=60, edgecolor="white", linewidth=0.3, color="steelblue")
ax.set_xlabel("Direct fan-in")
ax.set_ylabel("Count")
ax.set_title("Header Fan-In Distribution")
ax.set_yscale("log")

# Fan-out distribution for source files
ax = axes[1]
ax.hist(sources["direct_fan_out"], bins=60, edgecolor="white", linewidth=0.3, color="coral")
ax.set_xlabel("Direct fan-out (includes)")
ax.set_ylabel("Count")
ax.set_title("Source File Fan-Out Distribution")

fig.tight_layout()
plt.show()

In [ ]:
top_fanin = headers.sort_values("transitive_fan_in", ascending=False).head(20).copy()
top_fanin.index = range(1, len(top_fanin) + 1)

print("TOP 20 HEADERS BY TRANSITIVE FAN-IN")
print("=" * 90)
print(top_fanin[["file", "direct_fan_in", "transitive_fan_in"]].to_string())

In [ ]:
top_fanout = sources.sort_values("direct_fan_out", ascending=False).head(20).copy()
top_fanout.index = range(1, len(top_fanout) + 1)

print("TOP 20 SOURCE FILES BY DIRECT FAN-OUT")
print("=" * 80)
print(top_fanout[["file", "direct_fan_out"]].to_string())

## 3. PageRank

In [ ]:
pagerank_df = compute_header_pagerank(include_graph, exclude_system=True)

top_pr = pagerank_df.head(20).copy()
top_pr.index = range(1, len(top_pr) + 1)

print("TOP 20 HEADERS BY PAGERANK")
print("=" * 70)
print(top_pr.to_string())

In [ ]:
# Compare PageRank vs fan-in ranking
pr_ranked = pagerank_df.head(30).copy()
pr_ranked["pr_rank"] = range(1, len(pr_ranked) + 1)

fanin_ranked = headers.sort_values("transitive_fan_in", ascending=False).head(30).copy()
fanin_ranked["fanin_rank"] = range(1, len(fanin_ranked) + 1)

comparison = pr_ranked[["file", "pr_rank"]].merge(
    fanin_ranked[["file", "fanin_rank"]], on="file", how="outer"
).sort_values("pr_rank")
comparison.index = range(1, len(comparison) + 1)

print("PAGERANK vs FAN-IN RANKING (top 30 from each)")
print("=" * 60)
print(comparison.to_string())

# Count headers appearing in both top-20 lists
pr_top20_files = set(pagerank_df.head(20)["file"])
fanin_top20_files = set(headers.sort_values("transitive_fan_in", ascending=False).head(20)["file"])
overlap = pr_top20_files & fanin_top20_files
print(f"\nOverlap in top-20: {len(overlap)} / 20 headers appear in both lists")
print(f"Headers unique to PageRank top-20: {pr_top20_files - fanin_top20_files}")
print(f"Headers unique to fan-in top-20:   {fanin_top20_files - pr_top20_files}")

## 4. Include Amplification

In [ ]:
amplification_df = compute_include_amplification(include_graph, file_metrics=ds.file_metrics)

print(f"Source files with amplification data: {len(amplification_df):,}")
print(f"Mean amplification ratio:  {amplification_df['amplification_ratio'].mean():.1f}x")
print(f"Median amplification ratio: {amplification_df['amplification_ratio'].median():.1f}x")
print(f"Max amplification ratio:   {amplification_df['amplification_ratio'].max():.1f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(amplification_df["amplification_ratio"], bins=60, edgecolor="white", linewidth=0.3, color="teal")
ax.axvline(amplification_df["amplification_ratio"].median(), color="red", linestyle="--",
           alpha=0.7, label=f"Median ({amplification_df['amplification_ratio'].median():.1f}x)")
ax.set_xlabel("Amplification ratio (transitive / direct includes)")
ax.set_ylabel("Count")
ax.set_title("Include Amplification Distribution")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
top_amp = amplification_df.head(20).copy()
top_amp.index = range(1, len(top_amp) + 1)

print("TOP 20 SOURCE FILES BY AMPLIFICATION RATIO")
print("=" * 100)
print(top_amp.to_string())

In [ ]:
# Scatter: amplification_ratio vs compile_time_ms
fm = ds.file_metrics.copy()
if "source_file" in fm.columns:
    fm = fm.rename(columns={"source_file": "file"})

scatter_df = amplification_df.merge(
    fm[["file", "compile_time_ms"]].dropna(), on="file", how="inner"
)

if len(scatter_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(scatter_df["amplification_ratio"], scatter_df["compile_time_ms"],
              alpha=0.4, s=15, color="teal")
    ax.set_xlabel("Amplification ratio")
    ax.set_ylabel("Compile time (ms)")
    ax.set_title("Include Amplification vs Compile Time")
    fig.tight_layout()
    plt.show()

    corr = scatter_df[["amplification_ratio", "compile_time_ms"]].corr().iloc[0, 1]
    print(f"Pearson correlation: {corr:.3f}")
else:
    print("No matching files to produce scatter plot.")

## 5. Header Impact Scoring

In [ ]:
# Compute header churn from git data if available
if HAS_GIT:
    git_log = ds.git_commit_log
    header_churn_df = compute_file_churn(git_log)
    print(f"Git churn data: {len(header_churn_df):,} files")
else:
    header_churn_df = pd.DataFrame(columns=["source_file", "n_commits"])
    print("No git data — impact scores will not include churn weighting.")

In [ ]:
header_impact_df = compute_header_impact_score(
    fan_metrics=fan_metrics,
    header_metrics=ds.header_metrics,
    git_churn=header_churn_df,
)

top_impact = header_impact_df.head(30).copy()
top_impact.index = range(1, len(top_impact) + 1)

print("TOP 30 HEADERS BY IMPACT SCORE")
print("=" * 120)
print(top_impact.to_string())

In [ ]:
# Classify top headers by likely refactoring strategy
top30 = header_impact_df.head(30).copy()

def classify_header(row):
    strategies = []
    if row["transitive_fan_in"] > row["direct_fan_in"] * 2:
        strategies.append("forward declarations")
    if row["sloc"] > 500:
        strategies.append("split header")
    if row["sloc"] > 200 and row["transitive_fan_in"] > 10:
        strategies.append("pimpl")
    return ", ".join(strategies) if strategies else "review"

top30["refactoring_strategy"] = top30.apply(classify_header, axis=1)
top30.index = range(1, len(top30) + 1)

print("REFACTORING STRATEGY CLASSIFICATION")
print("=" * 130)
print(top30[["file", "impact_score", "transitive_fan_in", "sloc", "refactoring_strategy"]].to_string())

## 6. Generated Header Analysis

In [ ]:
# Determine generated vs handwritten headers using file_metrics
fm = ds.file_metrics
generated_files = set()
if "is_generated" in fm.columns and "source_file" in fm.columns:
    generated_files = set(fm.loc[fm["is_generated"], "source_file"])

# Also check header_metrics for origin hints
hm = ds.header_metrics
if "is_generated" in hm.columns:
    generated_files |= set(hm.loc[hm["is_generated"], "header_file"])

impact_with_origin = header_impact_df.copy()
impact_with_origin["origin"] = impact_with_origin["file"].apply(
    lambda f: "GENERATED" if f in generated_files else "HANDWRITTEN"
)

gen_headers = impact_with_origin[impact_with_origin["origin"] == "GENERATED"]
hw_headers = impact_with_origin[impact_with_origin["origin"] == "HANDWRITTEN"]

print(f"Generated headers with impact scores: {len(gen_headers):,}")
print(f"Handwritten headers with impact scores: {len(hw_headers):,}")

if len(gen_headers) > 0:
    print(f"\nGenerated headers — mean impact score:    {gen_headers['impact_score'].mean():,.0f}")
    print(f"Handwritten headers — mean impact score:  {hw_headers['impact_score'].mean():,.0f}")
    print(f"\nGenerated headers — mean fan-in:          {gen_headers['transitive_fan_in'].mean():.1f}")
    print(f"Handwritten headers — mean fan-in:        {hw_headers['transitive_fan_in'].mean():.1f}")

    # Top generated headers by impact
    top_gen = gen_headers.sort_values("impact_score", ascending=False).head(10).copy()
    top_gen.index = range(1, len(top_gen) + 1)
    print(f"\nTOP 10 GENERATED HEADERS BY IMPACT SCORE")
    print("=" * 100)
    print(top_gen[["file", "impact_score", "transitive_fan_in", "sloc", "n_commits"]].to_string())

    if gen_headers["transitive_fan_in"].mean() > hw_headers["transitive_fan_in"].mean():
        print("\n⚠ Generated headers have higher average fan-in than handwritten — codegen is creating coupling.")
else:
    print("No generated headers found in impact data.")

## 7. Gephi Export

In [ ]:
path = export_include_graph(
    include_graph=include_graph,
    header_metrics=ds.header_metrics,
    header_impact=header_impact_df,
    header_pagerank=pagerank_df,
    git_churn=header_churn_df,
    file_metrics=ds.file_metrics,
    output_path=INTERMEDIATE_DIR / "gephi" / "include_graph.gexf",
)
print(f"\nGEXF file: {path}")

### Gephi Usage Guide

**Setup:**
1. Open `data/intermediate/gephi/include_graph.gexf` in Gephi
2. Run **ForceAtlas2** layout (LinLog mode, scaling ~2.0, prevent overlap)
3. Size nodes by `impact_score` (or `pagerank`)
4. Colour nodes by `origin` (GENERATED vs HANDWRITTEN) or `cmake_target`
5. Use the Ranking panel to set edge thickness by `weight`

**What to look for:**
- **God headers:** Large, highly-connected central nodes with high `impact_score`
  — candidates for splitting or forward-declaration refactoring
- **Cross-target include bleeding:** Edges between nodes of different `cmake_target`
  values — indicates coupling between modules
- **Generated code clusters:** Nodes coloured as GENERATED forming dense subgraphs
  — codegen may be creating unnecessary coupling
- **High-amplification files:** Source nodes with many transitive includes but few
  direct — each `#include` pulls in a deep tree

In [ ]:
# Candidate nodes for Gephi investigation

print("CANDIDATE NODES FOR GEPHI INVESTIGATION")
print()

# Top headers by PageRank
print("--- Top 15 headers by PageRank ---")
for i, row in pagerank_df.head(15).iterrows():
    print(f"  {row['file']:<60s} pagerank={row['pagerank']:.6f}")

print()

# Top headers by impact score
print("--- Top 15 headers by impact score ---")
for i, (_, row) in enumerate(header_impact_df.head(15).iterrows()):
    print(f"  {row['file']:<60s} impact={row['impact_score']:>12,.0f}  "
          f"fan_in={row['transitive_fan_in']:>4d}  sloc={row['sloc']:>5d}  "
          f"commits={row['n_commits']:>4d}")

print()

# Highest amplification source files
print("--- Top 15 source files by amplification ratio ---")
for i, (_, row) in enumerate(amplification_df.head(15).iterrows()):
    print(f"  {row['file']:<60s} ratio={row['amplification_ratio']:>6.1f}x  "
          f"direct={row['direct_includes']:>3d}  transitive={row['transitive_includes']:>4d}")

## 8. Persist Output

In [ ]:
out_path = ds.save_intermediate("header_impact", header_impact_df)
print(f"Saved header_impact.parquet: {header_impact_df.shape[0]} rows x {header_impact_df.shape[1]} cols")
print(f"  -> {out_path}")